In [ ]:
# 1. --- UNIT PARSER ---
def parse_units(value):
    if pd.isna(value) or str(value).strip().lower() == "error" or str(value).strip() == "":
        return np.nan
    val_str = str(value).strip()
    match = re.search(r'([0-9\.-]+)([a-zA-Z]*)', val_str)
    if match:
        num = float(match.group(1))
        unit = match.group(2).lower()
        unit_map = {'f': 1e-15, 'p': 1e-12, 'n': 1e-9, 'u': 1e-6, 'm': 1e-3, 'k': 1e3, 'M': 1e6, 'G': 1e9}
        multiplier = unit_map.get(unit, 1.0)
        return num * multiplier
    return np.nan

# 2. --- LOADING DATA ---
file_max = "CC_max_PrimeSim_default_justCC_Measurements_history_1_20260410_16_41_15.9.csv"
file_min = "CC_min_PrimeSim_default_justCC_Measurements_history_1_20260410_16_56_40.57.csv"

df_max_raw = pd.read_csv(file_max)
df_min_raw = pd.read_csv(file_min)

# Parse Capacitance Columns
df_max_raw['Cap_val'] = df_max_raw['Cap_measured_max:ac'].apply(parse_units)
df_min_raw['Cap_val'] = df_min_raw['Cap_measured_min:ac'].apply(parse_units)

# 3. --- CONFIGURATION ---
target_corners = ['Typ25', 'bcQ-40', 'wcQ125']
colors = {'Typ25': 'green', 'bcQ-40': 'blue', 'wcQ125': 'red'}
labels = {'Typ25': 'Typical', 'bcQ-40': 'bcQ (Best)', 'wcQ125': 'wcQ (Worst)'}

# 4. --- PLOT 1: MINIMUM FREQUENCY STATE (C_max) ---
plt.figure(figsize=(10, 5))
for corner in target_corners:
    subset = df_max_raw[df_max_raw['Corner'] == corner].sort_values('Sweep:Variable:temp')
    if not subset.empty:
        plt.plot(subset['Sweep:Variable:temp'], subset['Cap_val'] * 1e12, 
                 color=colors[corner], marker='o', label=labels[corner])

plt.title("Coarse capacitance vs Temperature: Minimum Frequency State ($Cc_{max}$)", fontsize=13, fontweight='bold')
plt.xlabel("Temperature (°C)")
plt.ylabel("Capacitance (pF)")
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend()
plt.tight_layout()
plt.show()

# 5. --- PLOT 2: MAXIMUM FREQUENCY STATE (C_min) ---
plt.figure(figsize=(10, 5))
for corner in target_corners:
    subset = df_min_raw[df_min_raw['Corner'] == corner].sort_values('Sweep:Variable:temp')
    if not subset.empty:
        plt.plot(subset['Sweep:Variable:temp'], subset['Cap_val'] * 1e12, 
                 color=colors[corner], marker='s', label=labels[corner])

plt.title("Coarse capacitance vs Temperature: Maximum Frequency State ($Cc_{min}$)", fontsize=13, fontweight='bold')
plt.xlabel("Temperature (°C)")
plt.ylabel("Capacitance (pF)")
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend()
plt.tight_layout()
plt.show()

# Comparison at Temperature Extremes (-40C and 125C)
corners = ['Typ25', 'bcQ-40', 'wcQ125']

for state, df in [("C_max (Min Freq)", df_max_raw), ("C_min (Max Freq)", df_min_raw)]:
    print(f"\n--- {state} ---")
    for corner in corners:
        # Extract values (assuming Cap_val is already parsed)
        c40 = df[(df['Corner']==corner) & (df['Sweep:Variable:temp']==-40.0)]['Cap_val'].values[0]
        c125 = df[(df['Corner']==corner) & (df['Sweep:Variable:temp']==125.0)]['Cap_val'].values[0]
        
        delta_fF = (c40 - c125) * 1e15
        print(f"{corner}: C(-40) - C(125) = {delta_fF:.2f} fF")